# HadISD Tutorial Six - Processing with Parquet

> **NOTE**
> Before beginning this tutorial, you should first read [HadISD Tutorial One - Introduction to Station Data](./HadISD-One-Introduction.ipynb) and ensure you complete the tutorials in order

This notebook groups the stations similarly to notebook 3, but produces outputs in the Parquet format rather than NetCDF files. This is much more efficient later when retrieving data, and the conversion is also faster.

In [2]:
import tarfile
import gzip
import shutil
from pathlib import Path
import numpy as np
from datetime import datetime
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import os
import xarray as xr

In [3]:
UNPACKED_DIR = Path('/scratch/nf33/data/') / 'hadisd' / 'unpacked'       # We need a place on disk to unpack the archives
PROCESSING_DIR = Path('/scratch/nf33/data/') / 'hadisd' / 'processing'   # We need to cache some data on disk during reprocessing
PARQUET_DIR_JOB = Path('/scratch/nf33/data/') / 'hadisd' / 'parquet'

In [4]:
files = list(UNPACKED_DIR.glob('*.nc'))
len(files)

10393

In [5]:
def simplify(ds):
    '''
    Here we move the latitude, longitude and elevation from a coordinate to a data variable
    
    This fits the structure of the data more efficiently, allowing simpler concatenation.
    '''
    lats = xr.DataArray(data=np.full(len(ds.time), ds.latitude.values[0]), coords={'time': ds.time.values})
    lons = xr.DataArray(data=np.full(len(ds.time), ds.longitude.values[0]), coords={'time': ds.time.values})
    elev = xr.DataArray(data=np.full(len(ds.time), ds.elevation.values[0]), coords={'time': ds.time.values})
    year = xr.DataArray(data=np.full(len(ds.time), ds.time.dt.year), coords={'time': ds.time.values})
    ds = ds.reset_coords(names=('latitude', 'longitude', 'elevation'), drop=True)
    ds['lat'] = lats
    ds['lon'] = lons
    ds['elev'] = elev
    ds['year'] = year

    ds = ds.drop_attrs()
    ds = ds.drop_vars([
        "input_station_id",
        "quality_control_flags",
        "flagged_obs",
        "reporting_stats"
    ], errors="ignore")
    return ds

In [11]:
files = list(Path(UNPACKED_DIR).glob('*.nc'))
filegroups = [files[i:i + 100] for i in range(0, len(files), 100)]
print(len(filegroups))  # We come up with 134 such file groupings from the test data or 1040 for the full dataset

104


In [12]:
_decades = [('1800', '1930'), # Just in case there is undocumented early data
           ('1930', '1940'), ('1940', '1950'), # Dataset begins in 1930, start by decade here
           ('1950', '1960'), ('1960', '1970'), ('1970', '1980'), 
           ('1980', '1990'), ('1990', '2000'), ('2000', '2010'), # 1980 is a common time to start from
           ('2010', '2020'), ('2020', '2030')
          ]

In [13]:
# This takes around 20-60 seconds per grouping. If you just want to get the hang of it, limit it to three groupings
# Otherwise, the test set have 67 groupings, so will take around half an hour to run
# The full set of stations will take several hours!
# For testing, just try three file groups
decades = map(lambda x: _decades[x], [2,4,6])
import time
import pyarrow as pa
from tqdm import tqdm
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.dataset as pa_ds
from multiprocessing import Pool


def iter_batch(_files):
    for f in tqdm(_files):
        with xr.open_dataset(f, engine='netcdf4', chunks=None) as ds:
            ds = simplify(ds)
            yield pa.table(ds.to_dataframe())

def run_single_report(pool_arg):
    ts = time.time()
    report_id, fg = pool_arg
    ts = time.time()
    record_dir = f"{PARQUET_DIR_JOB}/report_{report_id}"
    os.makedirs(record_dir, exist_ok=True)
    pa_ds.write_dataset(
        pa.concat_tables(iter_batch(fg)).sort_by("time"),
        base_dir=record_dir,
        partitioning=["year"],
        format="parquet",
        existing_data_behavior="overwrite_or_ignore"
    )
    te = time.time()
    print(f"[parquet-group={report_id}] time taken total: {te - ts}s")
  
def parquet_processing():
    n_workers = 4
    with Pool(n_workers) as p:
        p.map(run_single_report, enumerate(filegroups))



In [14]:
%%time
# This uses up to 65GB of RAM for grouping sizes of 100, you may be able
# to run this on smaller machines by controlling the file group size and/or the
# number of workers
parquet_processing()

100%|██████████| 100/100 [00:40<00:00,  2.49it/s]


[parquet-group=7] time taken total: 43.72324585914612s


 20%|██        | 20/100 [00:06<00:23,  3.44it/s]

[parquet-group=0] time taken total: 50.971986055374146s


  1%|          | 1/100 [00:00<00:44,  2.22it/s]]

...

[parquet-group=89] time taken total: 49.49197030067444s


 35%|███▌      | 35/100 [00:12<00:23,  2.72it/s]

[parquet-group=103] time taken total: 43.45616912841797s


100%|██████████| 100/100 [00:39<00:00,  2.52it/s]


[parquet-group=97] time taken total: 45.09441018104553s
[parquet-group=90] time taken total: 56.166526317596436s
CPU times: user 7.55 s, sys: 3.4 s, total: 10.9 s
Wall time: 21min 56s


In [10]:
print('done')

done
